# UMAP by clinical outcome — rasterized dots, vector text

Colours the UMAP stored in `inprogress_3.h5ad` (`obsm['X_umap']`) by `obs['clinical_outcome']`
using the paper-wide convention:

| group | colour |
|---|---|
| **Good** | `#4DAF4A` |
| **Poor** | `#E41A1C` |

The saved `uns['outcome_colors']` palette in the file is the scanpy default blue/orange, so the
pair is hardcoded here instead of being read from the object.

Output: `outputs/images/processed_umap_clinical_outcome.svg` — the ~95k scatter dots are
**rasterized** into a single embedded image, while axes, title and legend stay editable vector
text for Illustrator/Inkscape.

The h5ad is opened strictly read-only (`h5py` in `'r'` mode); only `obs` and `obsm/X_umap` are
read, never `X` or `layers`, and nothing is written back to it.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc

# 'none' keeps every label as a real <text> element in the SVG instead of outlined paths
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
sc.set_figure_params(vector_friendly=True, dpi_save=600, frameon=False)

H5 = Path(f"{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad")
OUT = Path(f"{ACC_DATA_ROOT}/outputs/images")
SVG = OUT / "processed_umap_clinical_outcome.svg"

KEY = "clinical_outcome"
OUTCOME_COLOURS = {"Good": "#4DAF4A", "Poor": "#E41A1C"}  # ColorBrewer Set1 green / red

In [ ]:
# Slim read: obs + X_umap only. 'r' mode never writes to the file.
try:
    read_elem = ad.io.read_elem
except AttributeError:
    from anndata.experimental import read_elem

with h5py.File(H5, "r") as f:
    obs = read_elem(f["obs"])
    umap = read_elem(f["obsm/X_umap"])

adata = ad.AnnData(obs=obs, obsm={"X_umap": umap})

# Fix category order so the legend and palette line up Good-then-Poor
adata.obs[KEY] = adata.obs[KEY].cat.reorder_categories(["Good", "Poor"])
adata.uns[f"{KEY}_colors"] = [OUTCOME_COLOURS[c] for c in adata.obs[KEY].cat.categories]

print(f"{adata.n_obs:,} cells")
print(adata.obs[KEY].value_counts(dropna=False).to_string())
print("\npatients per group:")
print(adata.obs.groupby(KEY, observed=True)["batch"].nunique().to_string())

In [ ]:
# Shuffle the plotting order: the two groups are near-equal in size and heavily overlapping,
# so drawing them in file order would bury whichever sample block is written last.
rng = np.random.default_rng(0)
adata = adata[rng.permutation(adata.n_obs)].copy()

fig, ax = plt.subplots(figsize=(7, 6))

sc.pl.umap(
    adata,
    color=KEY,
    palette=OUTCOME_COLOURS,
    legend_loc="right margin",
    size=6,
    frameon=False,
    title="Clinical outcome",
    ax=ax,
    show=False,
)

# Rasterize the point cloud only. Collections are the scatter dots; text, spines and the
# legend handles are not collections, so they stay vector objects.
for coll in ax.collections:
    coll.set_rasterized(True)

# dpi controls the resolution of the rasterized dot layer inside the SVG
fig.savefig(SVG, dpi=600, bbox_inches="tight")
plt.close(fig)

print(f"{SVG}  ({SVG.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Sanity check: dots must be one embedded raster image, labels must still be <text>,
# and both outcome colours must appear in the vector legend swatches.
svg = SVG.read_text()
n_image = svg.count("<image")
n_text = svg.count("<text")
print(f"raster layers: {n_image}   editable <text>: {n_text}")
for label, hexcode in OUTCOME_COLOURS.items():
    present = hexcode.lower() in svg.lower()
    print(f"  {label:5s} {hexcode}  in SVG: {present}")
    assert present, f"{label} colour {hexcode} missing from the SVG"

assert n_image >= 1, "points were not rasterized"
assert n_text >= 1, "labels were outlined instead of kept as text"
print("\nOK — dots rasterized, text + legend vector, Good/Poor colours correct")

In [ ]:
# Confirm the source h5ad was untouched
import datetime
st = H5.stat()
print(H5.name, f"{st.st_size / 1e9:.2f} GB",
      "last modified", datetime.datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"))